# Chapter 7. Recording quality control

**Learning objective.** Diagnose whether a recorded file is technically readable and fit for a declared phonetic measure, while preserving the distinction between file integrity and measurement validity.

**Estimated time:** 45–60 minutes. **Prerequisite:** Chapter 7. **Exercises:** 7.1 and 7.2.

The bundled input is an explicitly **synthetic teaching fixture**: three one-second numerical tones created by `generate_ch07_fixture.py`. They are not speech, participant data, or evidence about a population. One file is intentionally clipped and one is intentionally near silent. To use authorized local study files, create a separate manifest with the documented columns and run the notebook locally; do not upload restricted recordings to Colab.

## Input and output contract

The input manifest must supply `file_id`, `session_id`, `source_path`, `expected_sample_rate_hz`, `expected_bit_depth`, `expected_channels`, and `data_status`. The notebook reads WAV headers and samples but never modifies audio. It writes `recording_qc.tsv`, `recording_qc_summary.html`, `run_provenance.json`, and `decision_record.json` under `companion/outputs/ch07/01_recording_qc/`.

A passing technical check is not a certificate of scientific validity. Measure-specific usability remains a research decision tied to the construct, recording chain, and planned estimator.

In [ ]:
from pathlib import Path
import csv
import json
import sys

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    if (candidate / "companion" / "data" / "MANIFEST.tsv").is_file():
        REPOSITORY_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the repository root or one of its subdirectories.")

SOURCE_ROOT = REPOSITORY_ROOT / "companion" / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

DATA_ROOT = REPOSITORY_ROOT / "companion" / "data" / "ch07" / "synthetic_recordings"
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "The Chapter 7 fixture is absent. Run companion/scripts/generate_ch07_fixture.py explicitly; "
        "the notebook will not create or substitute data silently."
    )
print(f"Repository: {REPOSITORY_ROOT}")
print("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — no participant observations are present.")

In [ ]:
from phonetic_research_companion.ch07 import run_recording_qc

MANIFEST = DATA_ROOT / "recording_manifest.tsv"
OUTPUT = REPOSITORY_ROOT / "companion" / "outputs" / "ch07" / "01_recording_qc"
results = run_recording_qc(MANIFEST, OUTPUT)

# Fixture invariants: these tests prevent an apparently successful but changed lesson.
assert len(results) == 3
assert sum("possible_digital_clipping" in row["warnings"] for row in results) == 1
assert sum("near_silent" in row["warnings"] for row in results) == 1
assert all(row["data_status"] == "SYNTHETIC_TEACHING_FIXTURE" for row in results)
print(f"Validated {len(results)} explicitly synthetic files; outputs: {OUTPUT}")

In [ ]:
display_fields = [
    "file_id", "qc_status", "peak_normalized", "rms_normalized", "clipped_fraction",
    "warnings", "usable_for_duration", "usable_for_relative_amplitude", "usable_for_spectral_measures",
]
for row in results:
    print({field: row.get(field) for field in display_fields})

## Learner decision

Explain why the clipped file may retain usable duration information while failing for relative-amplitude or spectral work. Then state whether each file should be accepted, re-recorded, restricted to named measures, or excluded. Your decision must cite the target measure and should not use a single global “good/bad audio” label.

In [ ]:
LEARNER_DECISION = {
    "data_status": "SYNTHETIC_TEACHING_FIXTURE",
    "decision_status": "undecided",
    "measure_specific_decisions": [],
    "rationale": "Learner response required before assessment.",
    "warning": "This clean-run record is not an empirical conclusion and must not be reported as one.",
}
(OUTPUT / "decision_record.json").write_text(json.dumps(LEARNER_DECISION, indent=2), encoding="utf-8")
assert (OUTPUT / "recording_qc.tsv").exists()
assert (OUTPUT / "recording_qc_summary.html").exists()
assert (OUTPUT / "run_provenance.json").exists()
print(json.dumps(LEARNER_DECISION, indent=2))